In [11]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import os
import copy
from sklearn.metrics import accuracy_score

Check Device (CPU/GPU)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


Define the GCN Model (CORE MODEL)

In [13]:
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim=32, output_dim=2):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x

Training Function (Reusable for Any Hospital)

In [14]:
def train_local_model(graph_path, epochs=100, lr=0.01):
    graph = torch.load(graph_path, weights_only=False).to(device)

    model = GCN(
        input_dim=graph.num_node_features,
        hidden_dim=32,
        output_dim=2
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model(graph.x, graph.edge_index)
        loss = criterion(out, graph.y)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"[{graph_path}] Epoch {epoch:03d} | Loss: {loss:.4f}")

    return model

In [15]:
# Evaluation Function
def evaluate_model(model, graph_path):
    graph = torch.load(graph_path, weights_only=False).to(device)
    model.eval()

    with torch.no_grad():
        logits = model(graph.x, graph.edge_index)
        preds = logits.argmax(dim=1).cpu()
        labels = graph.y.cpu()

    return accuracy_score(labels, preds)

Train Models at Each Hospital

In [16]:
model_A = train_local_model("../data/graph/graph_A.pt")
model_B = train_local_model("../data/graph/graph_B.pt")
model_C = train_local_model("../data/graph/graph_C.pt")

[../data/graph/graph_A.pt] Epoch 000 | Loss: 0.5923
[../data/graph/graph_A.pt] Epoch 020 | Loss: 0.0590
[../data/graph/graph_A.pt] Epoch 040 | Loss: 0.0470
[../data/graph/graph_A.pt] Epoch 060 | Loss: 0.0416
[../data/graph/graph_A.pt] Epoch 080 | Loss: 0.0429
[../data/graph/graph_B.pt] Epoch 000 | Loss: 0.5216
[../data/graph/graph_B.pt] Epoch 020 | Loss: 0.0796
[../data/graph/graph_B.pt] Epoch 040 | Loss: 0.0600
[../data/graph/graph_B.pt] Epoch 060 | Loss: 0.0575
[../data/graph/graph_B.pt] Epoch 080 | Loss: 0.0440
[../data/graph/graph_C.pt] Epoch 000 | Loss: 0.4638
[../data/graph/graph_C.pt] Epoch 020 | Loss: 0.0241
[../data/graph/graph_C.pt] Epoch 040 | Loss: 0.0108
[../data/graph/graph_C.pt] Epoch 060 | Loss: 0.0095
[../data/graph/graph_C.pt] Epoch 080 | Loss: 0.0043


In [17]:
print("Hospital A Accuracy (Clean):", evaluate_model(model_A, "../data/graph/graph_A.pt"))
print("Hospital B Accuracy (Clean):", evaluate_model(model_B, "../data/graph/graph_B.pt"))
print("Hospital C Accuracy (Clean):", evaluate_model(model_C, "../data/graph/graph_C.pt"))

Hospital A Accuracy (Clean): 0.990625
Hospital B Accuracy (Clean): 0.9829545454545454
Hospital C Accuracy (Clean): 1.0


In [18]:
# Save ONLY LDP-Protected Models
os.makedirs("../data/models", exist_ok=True)

torch.save(model_A_ldp.state_dict(), "../data/models/model_A.pth")
torch.save(model_B_ldp.state_dict(), "../data/models/model_B.pth")
torch.save(model_C_ldp.state_dict(), "../data/models/model_C.pth")

print("LDP-protected local models saved successfully")

LDP-protected local models saved successfully
